In [2]:
import pickle
pickle_path = "/Users/liam/quests/lsoc-psych/diversetm_eval_data/lsoc-evals/EleutherAI-pythia-6.9b_task_list_1.pkl"
with open(pickle_path, 'rb') as f:
    data = pickle.load(f)

data.keys()

dict_keys(['results', 'groups', 'group_subtasks', 'configs', 'versions', 'n-shot', 'higher_is_better', 'n-samples', 'samples', 'config', 'git_hash', 'date', 'pretty_env_info', 'transformers_version', 'lm_eval_version', 'upper_git_hash', 'tokenizer_pad_token', 'tokenizer_eos_token', 'tokenizer_bos_token', 'eot_token_id', 'max_length'])

In [13]:
data['configs']

{'anli_r1': {'task': 'anli_r1',
  'tag': ['anli'],
  'dataset_path': 'anli',
  'training_split': 'train_r1',
  'validation_split': 'dev_r1',
  'test_split': 'test_r1',
  'doc_to_text': '{{premise}}\nQuestion: {{hypothesis}} True, False, or Neither?\nAnswer:',
  'doc_to_target': "{{['True', 'Neither', 'False'][label]}}",
  'unsafe_code': False,
  'doc_to_choice': ['True', 'Neither', 'False'],
  'description': '',
  'target_delimiter': ' ',
  'fewshot_delimiter': '\n\n',
  'num_fewshot': 0,
  'metric_list': [{'metric': 'acc',
    'aggregation': 'mean',
    'higher_is_better': True}],
  'output_type': 'multiple_choice',
  'repeats': 1,
  'should_decontaminate': True,
  'doc_to_decontamination_query': 'premise',
  'metadata': {'version': 1.0}},
 'anli_r2': {'task': 'anli_r2',
  'tag': ['anli'],
  'dataset_path': 'anli',
  'training_split': 'train_r2',
  'validation_split': 'dev_r2',
  'test_split': 'test_r2',
  'doc_to_text': '{{premise}}\nQuestion: {{hypothesis}} True, False, or Neither?\

In [22]:
data['results']['drop']

{'alias': 'drop',
 'em,none': 0.001153523489932886,
 'em_stderr,none': 0.00034761798968569835,
 'f1,none': np.float64(0.031632760067114206),
 'f1_stderr,none': 0.0010368766196809577}

In [6]:
sorted(data['samples'].keys())

['anli_r1',
 'anli_r2',
 'anli_r3',
 'arc_easy',
 'cola',
 'drop',
 'gpqa_diamond_cot_n_shot',
 'gpqa_diamond_cot_zeroshot',
 'gpqa_diamond_generative_n_shot',
 'gpqa_diamond_n_shot',
 'gpqa_diamond_zeroshot',
 'gpqa_extended_cot_n_shot',
 'gpqa_extended_cot_zeroshot',
 'gpqa_extended_generative_n_shot',
 'gpqa_extended_n_shot',
 'gpqa_extended_zeroshot',
 'gpqa_main_cot_n_shot',
 'gpqa_main_cot_zeroshot',
 'gpqa_main_generative_n_shot',
 'gpqa_main_n_shot',
 'gpqa_main_zeroshot',
 'gsm8k',
 'hellaswag',
 'lambada_openai',
 'logiqa2',
 'mathqa',
 'mmlu_abstract_algebra',
 'mmlu_anatomy',
 'mmlu_astronomy',
 'mmlu_business_ethics',
 'mmlu_clinical_knowledge',
 'mmlu_college_biology',
 'mmlu_college_chemistry',
 'mmlu_college_computer_science',
 'mmlu_college_mathematics',
 'mmlu_college_medicine',
 'mmlu_college_physics',
 'mmlu_computer_security',
 'mmlu_conceptual_physics',
 'mmlu_econometrics',
 'mmlu_electrical_engineering',
 'mmlu_elementary_mathematics',
 'mmlu_formal_logic',
 'mm

In [8]:
data['samples']['anli_r1'][0]

{'doc_id': 0,
 'doc': {'uid': '4aae63a8-fcf7-406c-a2f3-50c31c5934a9',
  'premise': 'Ernest Jones is a British jeweller and watchmaker. Established in 1949, its first store was opened in Oxford Street, London. Ernest Jones specialises in diamonds and watches, stocking brands such as Gucci and Emporio Armani. Ernest Jones is part of the Signet Jewelers group.',
  'hypothesis': 'The first Ernest Jones store was opened on the continent of Europe.',
  'label': 0,
  'reason': "The first store was opened in London, which is in Europe. It may have been difficult for the system because continents weren't mentioned."},
 'target': 'True',
 'arguments': [('Ernest Jones is a British jeweller and watchmaker. Established in 1949, its first store was opened in Oxford Street, London. Ernest Jones specialises in diamonds and watches, stocking brands such as Gucci and Emporio Armani. Ernest Jones is part of the Signet Jewelers group.\nQuestion: The first Ernest Jones store was opened on the continent of 

In [ ]:
# For each anli_r1, anli_r2, anli_r3 we calculate the loss and accuracy as:
def calculate_metrics_per_question(data, eval_names):
    results = {}
    
    for eval_name in eval_names:
        if eval_name not in data['samples']:
            continue
            
        samples = data['samples'][eval_name]
        eval_results = []
        
        for sample in samples:
            # 1. Get accuracy (already calculated)
            accuracy = sample['acc']  # Always 0.0 or 1.0
            
            # 2. Find the correct answer's log-likelihood
            target = sample['target']
            target_log_likelihood = None
            
            for i, (prompt, answer) in enumerate(sample['arguments']):
                if answer.strip() == target:
                    target_log_likelihood = sample['filtered_resps'][i][0]
                    break
            
            # 3. Calculate loss from log-likelihood
            loss = -target_log_likelihood if target_log_likelihood is not None else None
            
            eval_results.append({
                'doc_id': sample['doc_id'],
                'accuracy': accuracy,  # 0.0 or 1.0
                'log_likelihood_on_answer_token': target_log_likelihood,
                'loss_on_answer_token': loss
            })
        
        results[eval_name] = eval_results
    
    return results

anli_evals = ['anli_r1', 'anli_r2', 'anli_r3']
results = calculate_metrics_per_question(data, anli_evals)

# Notes from asking claude things

- For each argument, the `[prompt, answer]` pair is run through the model for each tuple. That way it doesn't need to calculate the log-prob over all 50k tokens, just the three possible answers. 
- `'resps'` means "responses", and the log_likelihood value literally is just `loss= -log_likelihood`. Der. 
- The binary value in `resps[0][0] = (-1.544921875, True)` just means greedy decoding, which says "take the answer with the highest probability". 
- Filtering responses can mean other processing like removing invalid responses and stuff, but here it just flattens the list. 
- Oh shit this process is actually awful if model's tokenise things differently. God damn awful!!! Need to extract the answer from some number of tokens, like a number from "The answer is 150.". In this case I guess it's just assumed that True and False are tokens for all mdoels? `Neither` might not be though, hm. 
- `acc` is always `0.0, 1.0`. 

### For `arc_easy`:
- The full phrase `' Sunlight is the source of energy for nearly all ecosystems.'` gets put into the model and the log-likelihood in resps is the _total_ log likelihood no matter the columns. 
- The label is just about compatability with the original eval. It's not something that gets fed into the model. 
- In this case "target" is already the correct index. In the case of `anli` target was a string which had to be compared to the answers.

In [14]:
data['samples']['arc_easy'][0]

{'doc_id': 0,
 'doc': {'id': 'Mercury_417466',
  'question': 'Which statement best explains why photosynthesis is the foundation of most food webs?',
  'choices': {'text': ['Sunlight is the source of energy for nearly all ecosystems.',
    'Most ecosystems are found on land instead of in water.',
    'Carbon dioxide is more available than other gases.',
    'The producers in all ecosystems are plants.'],
   'label': ['A', 'B', 'C', 'D']},
  'answerKey': 'A'},
 'target': 0,
 'arguments': [('Question: Which statement best explains why photosynthesis is the foundation of most food webs?\nAnswer:',
   ' Sunlight is the source of energy for nearly all ecosystems.'),
  ('Question: Which statement best explains why photosynthesis is the foundation of most food webs?\nAnswer:',
   ' Most ecosystems are found on land instead of in water.'),
  ('Question: Which statement best explains why photosynthesis is the foundation of most food webs?\nAnswer:',
   ' Carbon dioxide is more available than ot

In [15]:
data['configs']['arc_easy']

{'task': 'arc_easy',
 'tag': ['ai2_arc'],
 'dataset_path': 'allenai/ai2_arc',
 'dataset_name': 'ARC-Easy',
 'training_split': 'train',
 'validation_split': 'validation',
 'test_split': 'test',
 'doc_to_text': 'Question: {{question}}\nAnswer:',
 'doc_to_target': '{{choices.label.index(answerKey)}}',
 'unsafe_code': False,
 'doc_to_choice': '{{choices.text}}',
 'description': '',
 'target_delimiter': ' ',
 'fewshot_delimiter': '\n\n',
 'num_fewshot': 0,
 'metric_list': [{'metric': 'acc',
   'aggregation': 'mean',
   'higher_is_better': True},
  {'metric': 'acc_norm', 'aggregation': 'mean', 'higher_is_better': True}],
 'output_type': 'multiple_choice',
 'repeats': 1,
 'should_decontaminate': True,
 'doc_to_decontamination_query': 'Question: {{question}}\nAnswer:',
 'metadata': {'version': 1.0}}

In [ ]:
# This version works for both anli and arc_easy
def calculate_metrics_universal(data, eval_names):
    results = {}
    
    for eval_name in eval_names:
        if eval_name not in data['samples']:
            continue
            
        samples = data['samples'][eval_name]
        eval_results = []
        
        for sample in samples:
            accuracy = sample['acc']
            
            # Handle both string targets (ANLI) and index targets (ARC)
            if isinstance(sample['target'], str):
                # ANLI format: find matching string
                target_log_likelihood = None
                for i, (prompt, answer) in enumerate(sample['arguments']):
                    if answer.strip() == sample['target']:
                        target_log_likelihood = sample['filtered_resps'][i][0]
                        break
            else:
                # ARC format: use index directly
                target_index = sample['target']
                target_log_likelihood = sample['filtered_resps'][target_index][0]
            
            loss = -target_log_likelihood if target_log_likelihood is not None else None
            
            eval_results.append({
                'doc_id': sample['doc_id'],
                'accuracy': accuracy,
                'log_likelihood_on_answer_token': target_log_likelihood,
                'loss_on_answer_token': loss,
                'target': sample['target']
            })
        
        results[eval_name] = eval_results
    
    return results

targets_are_strings = ['anli_r1', 'anli_r2', 'anli_r3']
targets_are_indices = ['arc_easy', 'cola']

evals_mcc = ['cola']

### For cola (Compose Objects Localised with Attributes):
- The data uses MCC (Matthews Correlation Coefficient / Phi coefficient) instead of straight accuracy. This is because of class imbalance problems in binary classification. If you have 90% of the answers being "Yes (the sentence makes sense) then the model might show 90% accuracy just by guessing that answer each time. 
- So the log-likelihoods are still relevant, but the "overall accurracy / MCC" requires the data of `(predicted, target)` where both of these are the respective indices. 
   Then calculate:
```
mcc = (target, predicted)

TP = count((1,1))  # True Positives
TN = count((0,0))  # True Negatives
FP = count((0,1))  # False Positives  
FN = count((1,0))  # False Negatives

final_mcc = (TP*TN - FP*FN) / sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
```
- For binary classification this means `mcc` is `[-1, 1]`. Which is not `[0,1]` which is v important! 
- Note that `mcc = (target, predicted)`. So in the above example, the target is 1 but the model predicts 0 with lower log-likelihood (higher nll)
- In the end, I already have the cola result, I would only need to recalculate this if I was doing a subsetted aggregation over questions in cola. So I can just include it in the list like arc_easy (target is an index)

In [19]:
data['samples']['cola'][1]

{'doc_id': 1,
 'doc': {'sentence': 'The weights made the rope stretch over the pulley.',
  'label': 1,
  'idx': 1},
 'target': 1,
 'arguments': [('The weights made the rope stretch over the pulley.\nQuestion: Does this sentence make sense?\nAnswer:',
   ' no'),
  ('The weights made the rope stretch over the pulley.\nQuestion: Does this sentence make sense?\nAnswer:',
   ' yes')],
 'resps': [[(-4.87890625, False)], [(-5.01171875, False)]],
 'filtered_resps': [(-4.87890625, False), (-5.01171875, False)],
 'filter': 'none',
 'metrics': ['mcc'],
 'doc_hash': '04fbabe6648410d87e5f0059f4aa817d3c2ebca5f4eb3405367a6efc6c9b7029',
 'prompt_hash': '8a56d9bec5190fc60baf4a9047d02be029790bb27bb2058268f709f747c36111',
 'target_hash': '6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b',
 'mcc': (1, np.int64(0))}

In [18]:
data['configs']['cola']

{'task': 'cola',
 'tag': 'glue',
 'dataset_path': 'glue',
 'dataset_name': 'cola',
 'training_split': 'train',
 'validation_split': 'validation',
 'doc_to_text': '{{sentence}}\nQuestion: Does this sentence make sense?\nAnswer:',
 'doc_to_target': 'label',
 'unsafe_code': False,
 'doc_to_choice': ['no', 'yes'],
 'description': '',
 'target_delimiter': ' ',
 'fewshot_delimiter': '\n\n',
 'num_fewshot': 0,
 'metric_list': [{'metric': 'mcc'}],
 'output_type': 'multiple_choice',
 'repeats': 1,
 'should_decontaminate': True,
 'doc_to_decontamination_query': 'sentence',
 'metadata': {'version': 1.0}}

### DROP (Discrete Reasoning over Paragraphs) is completely different: 
- It is a "generate_until" dataset, where it goes until `{'until': ['.']}` 
- `em` means Exact Match, and `f1` means F1 score. 
- Exact match does what it says on the tin: it needs the exact string
```
model_answer = "Week 17: at Kansas City Chiefs..."
valid_answers = ["Chaz Schilens", "JaMarcus Russell"]
exact_match = 0  # No match
```
- I found the [code for calculating em and f1](https://github.com/EleutherAI/lm-evaluation-harness/blob/fcddf195ec6bb69c63e36d54d75354f6ecaabab7/lm_eval/tasks/drop/utils.py#L75). 
- It normalises as follows: 
```
def _normalize(answer):
    tokens = [
        _white_space_fix(_remove_articles(_fix_number(_remove_punc(token.lower()))))
        for token in _tokenize(answer)
    ]
    tokens = [token for token in tokens if token.strip()]
    normalized = " ".join(tokens).strip()
    return normalized
```
- Which means it removes ["the", "an", "a"] (articles), it reduces everything to lower case, removes punctuation, and removes some whitespace. 
- Exact match then requires that the normalised string of `predicted` is an _exact_ match for `gold` (gold meaning "gold" answers, of which there could be a few; in this case `['Chaz Schilens'], ['JaMarcus Russell']`).
- EM is binary [0,1]
- F1 score is a bit more forgiving and compares "bags" of words, where order doesn't matter. 
  - So F1=1.0 could be `predicted_bag = {"schilens", "chaz"}, gold_bag = {"chaz", "schilens"}`.
- Meanwhile, `predicted_bag = {"chaz", "schilens", "scored", "the", "touchdown"}, gold_bag = {"chaz", "schilens"}` would be F1=0.57.
- There is a calculation about precision and recall that happens. 
```
def _compute_f1(predicted_bag, gold_bag):
    intersection = len(gold_bag.intersection(predicted_bag))
    
    if not predicted_bag:
        precision = 1.0  # Edge case: empty prediction
    else:
        precision = intersection / float(len(predicted_bag))
    
    if not gold_bag:
        recall = 1.0  # Edge case: empty gold
    else:
        recall = intersection / float(len(gold_bag))
    
    f1 = (2 * precision * recall) / (precision + recall) if not (precision == 0.0 and recall == 0.0) else 0.0
    return f1
```

In [21]:
data['samples']['drop'][0]

{'doc_id': 0,
 'doc': {'section_id': 'nfl_1184',
  'passage': " Hoping to rebound from their loss to the Patriots, the Raiders stayed at home for a Week 16 duel with the Houston Texans.  Oakland would get the early lead in the first quarter as quarterback JaMarcus Russell completed a 20-yard touchdown pass to rookie wide receiver Chaz Schilens.  The Texans would respond with fullback Vonta Leach getting a 1-yard touchdown run, yet the Raiders would answer with kicker Sebastian Janikowski getting a 33-yard and a 30-yard field goal.  Houston would tie the game in the second quarter with kicker Kris Brown getting a 53-yard and a 24-yard field goal. Oakland would take the lead in the third quarter with wide receiver Johnnie Lee Higgins catching a 29-yard touchdown pass from Russell, followed up by an 80-yard punt return for a touchdown.  The Texans tried to rally in the fourth quarter as Brown nailed a 40-yard field goal, yet the Raiders' defense would shut down any possible attempt.",
  '

In [20]:
data['configs']['drop']

{'task': 'drop',
 'dataset_path': 'EleutherAI/drop',
 'dataset_kwargs': {'trust_remote_code': True},
 'training_split': 'train',
 'validation_split': 'validation',
 'process_docs': 'def process_docs(dataset):\n    def _process(doc):\n        return {\n            "id": doc["query_id"],\n            "passage": doc["passage"],\n            "question": doc["question"],\n            "answers": get_answers(doc),\n        }\n\n    return dataset.map(_process)\n',
 'doc_to_text': '{{passage}} {{question}}',
 'doc_to_target': "{{ answer|join(',')}}",
 'unsafe_code': False,
 'process_results': 'def process_results(doc, results):\n    preds, golds = results, doc["answers"]\n    max_em = 0\n    max_f1 = 0\n    for gold_answer in golds:\n        exact_match, f1_score = get_metrics(preds, gold_answer)\n        if gold_answer[0].strip():\n            max_em = max(max_em, exact_match)\n            max_f1 = max(max_f1, f1_score)\n    return {"em": max_em, "f1": max_f1}\n',
 'description': '',
 'target

### GPQA

- Okay so the CoT ones here prompt the model with `"Let's think step by step:"` at the end. 
- I have discovered that the `gpqa_{}_n_shot` evals are actually set with `num_fewshot=0`, so they only differ in prompting: 

```
gpqa_main_n_shot[0]
> "Here are some example questions from experts. Answer the final question yourself, following the format of the previous questions exactly.

{null}

Question: [question]
Choices:
(A) R-loops
(B) antisense
(C) lariat
(D) polyA tail
Let's think step by step: "
```
versus 
```
gpqa_main_zeroshot[0]
> "What is the correct answer to this question:[question]
Choices:
(A) R-loops
(B) antisense
(C) lariat
(D) polyA tail
Let's think step by step: "
```
- I found out the eval scores, despite being the same questions under the hood, are actually not the same between `n_shot` and `zero_shot`, probably due to the model's stochasticity as well as the subtle prompting difference (as well as the fact it might be handicappted by expecting to see multiple examples and seeing nothing).
- The model is scored on whether the string `"The answer is (A)"` (strict pattern) or `{(letter)}` (e.g. `(A), (B), (C), (D)`) is in the output string which includes CoT (flexible pattern). The answer is taken to be the last such string if there are multiple. If neither of these are in its output, the answer is marked as `[invalid]` (RIP).
- Then `main`, `extended` and `diamond` are all just different sets of questions, I think in increasing order of difficulty. 


| Variant | Few-Shot Examples | Prompt Ending | Expected Response | Stop Tokens | Metrics |
|---------|-------------------|---------------|-------------------|-------------|---------|
| **gpqa_main_zeroshot** | None | N/A (loglikelihood) | N/A | N/A | `acc`, `acc_norm` |
| **gpqa_main_n_shot** | Yes* | N/A (loglikelihood) | N/A | N/A | `acc`, `acc_norm` |
| **gpqa_main_cot_zeroshot** | None | `Let's think step by step:` | `Reasoning + (A)` | `['</s>']` | `exact_match` |
| **gpqa_main_cot_n_shot** | Yes* | `Let's think step by step:` | `Reasoning + (A)` | `['</s>']` | `exact_match` |
| **gpqa_main_generative_n_shot** | Yes* | `Answer:` | `(A)` | `['</s>', 'Question:', '<im_end>']` | `exact_match` |

*Note: All variants currently have `num_fewshot=0`, but the n_shot versions include priming text suggesting examples exist.

## Exact Prompts Used

**gpqa_main_zeroshot (Multiple Choice Loglikelihood):**
Question: [question]
Choices:
(A) [choice1]
(B) [choice2]
(C) [choice3]
(D) [choice4]
Answer:
*Evaluates loglikelihood of " (A)", " (B)", " (C)", " (D)" tokens*

**gpqa_main_n_shot (Multiple Choice Loglikelihood):**
Here are some example questions from experts. Answer the final question yourself, following the format of the previous questions exactly.
Question: [question]
Choices:
(A) [choice1]
(B) [choice2]
(C) [choice3]
(D) [choice4]
Answer:
*Evaluates loglikelihood of " (A)", " (B)", " (C)", " (D)" tokens*

**gpqa_main_cot_zeroshot:**
What is the correct answer to this question:[question]
Choices:
(A) [choice1]
(B) [choice2]
(C) [choice3]
(D) [choice4]
Let's think step by step:

**gpqa_main_cot_n_shot:**
Here are some example questions from experts. Answer the final question yourself, following the format of the previous questions exactly.
Question: [question]
Choices:
(A) [choice1]
(B) [choice2]
(C) [choice3]
(D) [choice4]
Let's think step by step:

**gpqa_main_generative_n_shot:**
Here are some example questions from experts. Answer the final question yourself, following the format of the previous questions exactly.
Question: [question]
Choices:
(A) [choice1]
(B) [choice2]
(C) [choice3]
(D) [choice4]
Answer:

## Evaluation Details
- **Loglikelihood variants**: Compare P(" (A)"), P(" (B)"), P(" (C)"), P(" (D)") and pick highest
- **Generative variants**: Two-stage regex (strict-match → flexible-extract) to extract final answer choice
- **Target**: Letter choice like `"(A)"`, `"(B)"`, etc.

In [ ]:
GPQA = ['gpqa_diamond_cot_n_shot',
 'gpqa_diamond_cot_zeroshot',
 'gpqa_diamond_generative_n_shot',
 'gpqa_diamond_n_shot',
 'gpqa_diamond_zeroshot',
 'gpqa_extended_cot_n_shot',
 'gpqa_extended_cot_zeroshot',
 'gpqa_extended_generative_n_shot',
 'gpqa_extended_n_shot',
 'gpqa_extended_zeroshot',
 'gpqa_main_cot_n_shot',
 'gpqa_main_cot_zeroshot',
 'gpqa_main_generative_n_shot',
 'gpqa_main_n_shot',
 'gpqa_main_zeroshot']

In [52]:
data['samples']['gpqa_main_zeroshot'][0]

{'doc_id': 0,
 'doc': {'Pre-Revision Question': "A large gene has dozens of exons, of which the central ones code for folded triple helical repeats that connect the cytoskeleton with sarcolemma and extracellular space. Each exon usually codes for one folded triple alpha helix. The most common mutations of the gene are central exon deletions that create out-of-frame peptides and progressive degenerative organ waste. A solution is to deliver a Morpholino that recognizes the 5' end of the out-of-frame exon in pre-mRNA. The molecule prevents binding of the spliceosome and creates exon skipping and in-frame joining. Several missing exons are well tolerated by an organism. Which structure below is not involved in the proposed therapy?",
  'Pre-Revision Correct Answer': 'R-loops',
  'Pre-Revision Incorrect Answer 1': 'lariat',
  'Pre-Revision Incorrect Answer 2': 'poly(A) tail',
  'Pre-Revision Incorrect Answer 3': 'antisense',
  'Pre-Revision Explanation': "The text describes the dystrophin 

In [53]:
data['samples']['gpqa_main_n_shot'][0]

{'doc_id': 0,
 'doc': {'Pre-Revision Question': "A large gene has dozens of exons, of which the central ones code for folded triple helical repeats that connect the cytoskeleton with sarcolemma and extracellular space. Each exon usually codes for one folded triple alpha helix. The most common mutations of the gene are central exon deletions that create out-of-frame peptides and progressive degenerative organ waste. A solution is to deliver a Morpholino that recognizes the 5' end of the out-of-frame exon in pre-mRNA. The molecule prevents binding of the spliceosome and creates exon skipping and in-frame joining. Several missing exons are well tolerated by an organism. Which structure below is not involved in the proposed therapy?",
  'Pre-Revision Correct Answer': 'R-loops',
  'Pre-Revision Incorrect Answer 1': 'lariat',
  'Pre-Revision Incorrect Answer 2': 'poly(A) tail',
  'Pre-Revision Incorrect Answer 3': 'antisense',
  'Pre-Revision Explanation': "The text describes the dystrophin 

In [26]:
data['samples']['gpqa_main_cot_n_shot'][0]

{'doc_id': 0,
 'doc': {'Pre-Revision Question': "A large gene has dozens of exons, of which the central ones code for folded triple helical repeats that connect the cytoskeleton with sarcolemma and extracellular space. Each exon usually codes for one folded triple alpha helix. The most common mutations of the gene are central exon deletions that create out-of-frame peptides and progressive degenerative organ waste. A solution is to deliver a Morpholino that recognizes the 5' end of the out-of-frame exon in pre-mRNA. The molecule prevents binding of the spliceosome and creates exon skipping and in-frame joining. Several missing exons are well tolerated by an organism. Which structure below is not involved in the proposed therapy?",
  'Pre-Revision Correct Answer': 'R-loops',
  'Pre-Revision Incorrect Answer 1': 'lariat',
  'Pre-Revision Incorrect Answer 2': 'poly(A) tail',
  'Pre-Revision Incorrect Answer 3': 'antisense',
  'Pre-Revision Explanation': "The text describes the dystrophin 

In [35]:
data['samples']['gpqa_main_cot_zeroshot'][0]

{'doc_id': 0,
 'doc': {'Pre-Revision Question': "A large gene has dozens of exons, of which the central ones code for folded triple helical repeats that connect the cytoskeleton with sarcolemma and extracellular space. Each exon usually codes for one folded triple alpha helix. The most common mutations of the gene are central exon deletions that create out-of-frame peptides and progressive degenerative organ waste. A solution is to deliver a Morpholino that recognizes the 5' end of the out-of-frame exon in pre-mRNA. The molecule prevents binding of the spliceosome and creates exon skipping and in-frame joining. Several missing exons are well tolerated by an organism. Which structure below is not involved in the proposed therapy?",
  'Pre-Revision Correct Answer': 'R-loops',
  'Pre-Revision Incorrect Answer 1': 'lariat',
  'Pre-Revision Incorrect Answer 2': 'poly(A) tail',
  'Pre-Revision Incorrect Answer 3': 'antisense',
  'Pre-Revision Explanation': "The text describes the dystrophin 

In [55]:
data['samples']['gpqa_main_generative_n_shot'][0]

{'doc_id': 0,
 'doc': {'Pre-Revision Question': "A large gene has dozens of exons, of which the central ones code for folded triple helical repeats that connect the cytoskeleton with sarcolemma and extracellular space. Each exon usually codes for one folded triple alpha helix. The most common mutations of the gene are central exon deletions that create out-of-frame peptides and progressive degenerative organ waste. A solution is to deliver a Morpholino that recognizes the 5' end of the out-of-frame exon in pre-mRNA. The molecule prevents binding of the spliceosome and creates exon skipping and in-frame joining. Several missing exons are well tolerated by an organism. Which structure below is not involved in the proposed therapy?",
  'Pre-Revision Correct Answer': 'R-loops',
  'Pre-Revision Incorrect Answer 1': 'lariat',
  'Pre-Revision Incorrect Answer 2': 'poly(A) tail',
  'Pre-Revision Incorrect Answer 3': 'antisense',
  'Pre-Revision Explanation': "The text describes the dystrophin 

In [27]:
data['configs']['gpqa_main_cot_n_shot']

{'task': 'gpqa_main_cot_n_shot',
 'tag': 'gpqa',
 'dataset_path': 'Idavidrein/gpqa',
 'dataset_name': 'gpqa_main',
 'training_split': 'train',
 'validation_split': 'train',
 'process_docs': 'def process_docs(dataset: datasets.Dataset) -> datasets.Dataset:\n    def _process_doc(doc):\n        choices = [\n            preprocess(doc["Incorrect Answer 1"]),\n            preprocess(doc["Incorrect Answer 2"]),\n            preprocess(doc["Incorrect Answer 3"]),\n            preprocess(doc["Correct Answer"]),\n        ]\n\n        random.shuffle(choices)\n        correct_answer_index = choices.index(preprocess(doc["Correct Answer"]))\n\n        out_doc = {\n            "choice1": choices[0],\n            "choice2": choices[1],\n            "choice3": choices[2],\n            "choice4": choices[3],\n            "choices": [choices[0], choices[1], choices[2], choices[3]],\n            "answer": f"({chr(65 + correct_answer_index)})",\n        }\n        return out_doc\n\n    return dataset.map(

In [39]:
gpqa_evals = [key for key in data['samples'].keys() if key.startswith('gpqa_')]

for eval_name in gpqa_evals:
    if 'num_fewshot' in data['configs'][eval_name].keys():
        num_fewshots = data['configs'][eval_name]['num_fewshot']
    else:
        num_fewshots = 'not there'
    print(f"{eval_name} fewshot: {num_fewshots}")


gpqa_diamond_cot_n_shot fewshot: 0
gpqa_extended_cot_n_shot fewshot: 0
gpqa_main_cot_n_shot fewshot: 0
gpqa_diamond_cot_zeroshot fewshot: 0
gpqa_extended_cot_zeroshot fewshot: 0
gpqa_main_cot_zeroshot fewshot: 0
gpqa_diamond_generative_n_shot fewshot: 0
gpqa_extended_generative_n_shot fewshot: 0
gpqa_main_generative_n_shot fewshot: 0
gpqa_diamond_n_shot fewshot: 0
gpqa_extended_n_shot fewshot: 0
gpqa_main_n_shot fewshot: 0
gpqa_diamond_zeroshot fewshot: 0
gpqa_extended_zeroshot fewshot: 0
gpqa_main_zeroshot fewshot: 0


In [61]:
import pandas as pd

eval_losses = pd.read_csv('evals_aggregate_n58.csv', index_col=0)
gpqa_data = eval_losses.loc[:, gpqa_evals]
gpqa_data = gpqa_data.sort_index()

# Make plotly heatmap 
import plotly.graph_objects as go
fig = go.Figure(data=go.Heatmap(
        z=gpqa_data.values.T,
        x=gpqa_data.index,  # evaluations on x-axis
        y=gpqa_data.columns,    # models on y-axis (ordered by PC)
        colorscale='RdBu_r',
        hoverongaps=False,
        zmid=0.5,
        colorbar=dict(title='GPQA scores', title_side='right')
    ))
fig.update_layout(
        title=f'Raw GPQA Data for n=58',
        xaxis_title='Evaluations',
        yaxis_title='Models',
        height=800,
        width=1500
    )

fig.show()

fig.write_image('/Users/liam/quests/lsoc-psych/lsoc1/figures/aggregate_pca_v1/gpqa_heatmap_n58.png', scale=2)

In [51]:
# number of samples in each gpqa eval
eval_sample_dict = {}
for eval_name in gpqa_evals:
    num_samples = len(data['samples'][eval_name]) if eval_name in data['samples'] else 0
    eval_sample_dict[eval_name] = num_samples
    # sort names alphabetically

eval_sample_dict = dict(sorted(eval_sample_dict.items()))
for eval_name, num_samples in eval_sample_dict.items():

    print(f"{eval_name} has {num_samples} samples")

gpqa_diamond_cot_n_shot has 396 samples
gpqa_diamond_cot_zeroshot has 396 samples
gpqa_diamond_generative_n_shot has 396 samples
gpqa_diamond_n_shot has 198 samples
gpqa_diamond_zeroshot has 198 samples
gpqa_extended_cot_n_shot has 1092 samples
gpqa_extended_cot_zeroshot has 1092 samples
gpqa_extended_generative_n_shot has 1092 samples
gpqa_extended_n_shot has 546 samples
gpqa_extended_zeroshot has 546 samples
gpqa_main_cot_n_shot has 896 samples
gpqa_main_cot_zeroshot has 896 samples
gpqa_main_generative_n_shot has 896 samples
gpqa_main_n_shot has 448 samples
gpqa_main_zeroshot has 448 samples
